# J2C-TST-EN: results by social platform

Subgroup evaluation of every English baseline by target social platform. Per-example metrics are joined to the official corpus through `row_id`; the notebook reports uncertainty, rankings, platform gaps, and multiplicity-adjusted tests.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy import stats

LANGUAGE = "EN"
RANDOM_SEED = 42
N_BOOTSTRAP = 2000

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "4-official_corpora").is_dir() and (candidate / "6-evaluation").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Cannot locate the J2C-TST project root")

PROJECT_ROOT = find_project_root()
CORPUS_PATH = PROJECT_ROOT / "4-official_corpora" / LANGUAGE / f"J2C_TST_{LANGUAGE}.csv"
RESULTS_DIR = PROJECT_ROOT / "6-evaluation" / f"J2C-TST-{LANGUAGE}" / "results"
NOTEBOOK_DIR = PROJECT_ROOT / "8-results_analysis" / f"J2C-TST-{LANGUAGE}"
ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="paper", style="whitegrid", palette="colorblind")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titleweight": "bold",
    "font.size": 10,
})

print(f"Corpus: {CORPUS_PATH}")
print(f"Evaluation results: {RESULTS_DIR}")
print(f"Artifacts: {ARTIFACTS_DIR}")

## 1. Join evaluation outputs to platform labels

In [ ]:
corpus = pd.read_csv(CORPUS_PATH)
corpus.insert(0, "row_id", np.arange(len(corpus), dtype=int))
test_lookup = corpus.loc[corpus["split"] == "test", ["row_id", "social_network"]].copy()

metric_files = sorted(RESULTS_DIR.glob("*/per_example_metrics.csv"))
if not metric_files:
    raise FileNotFoundError(f"No per-example metric files found under {RESULTS_DIR}")

frames = []
for metric_file in metric_files:
    system_id = metric_file.parent.name
    frame = pd.read_csv(metric_file)
    frame["system_id"] = system_id

    entity_file = metric_file.parent / "entities" / "entity_metrics_per_example.csv"
    if not entity_file.is_file():
        raise FileNotFoundError(f"Missing entity metrics for {system_id}")
    entity = pd.read_csv(entity_file)
    frame = frame.merge(entity, on="row_id", how="left", validate="one_to_one")
    frames.append(frame)

results = pd.concat(frames, ignore_index=True)
results = results.merge(test_lookup, on="row_id", how="left", validate="many_to_one")
if results["social_network"].isna().any():
    missing_ids = results.loc[results["social_network"].isna(), "row_id"].unique()[:10]
    raise ValueError(f"Metrics could not be mapped to the official test split: {missing_ids}")

results["model"] = np.where(
    results["model_id"].str.contains("gemma", case=False, na=False), "Gemma", "Llama"
)
results["strategy"] = results["strategy"].str.replace("_", " ").str.title()
results["system"] = results["model"] + " / " + results["strategy"]

def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    return series.astype(str).str.lower().map({"true": True, "false": False, "1": True, "0": False})

for column in ["style_target_correct", "all_editorial_constraints_ok"]:
    results[column] = as_bool(results[column])
    if results[column].isna().any():
        raise ValueError(f"Invalid boolean values in {column}")

expected_per_system = len(test_lookup)
coverage = (
    results.groupby("system_id", observed=True)
    .agg(rows=("row_id", "size"), unique_rows=("row_id", "nunique"), platforms=("social_network", "nunique"))
    .reset_index()
)
coverage["complete"] = (coverage["rows"] == expected_per_system) & (coverage["unique_rows"] == expected_per_system)
if not coverage["complete"].all():
    raise ValueError("At least one system is incomplete; inspect the coverage table")

display(coverage)
display(pd.crosstab(test_lookup["social_network"], columns="test examples"))

## 2. Platform-specific performance

In [ ]:
grouped = (
    results.groupby(["social_network", "system"], observed=True)
    .agg(
        n=("row_id", "size"),
        style_accuracy=("style_target_correct", "mean"),
        editorial_compliance=("all_editorial_constraints_ok", "mean"),
        bertscore_f1=("bertscore_f1_source", "mean"),
        sentence_bleu=("sentence_bleu_reference", "mean"),
        perplexity_median=("perplexity", "median"),
        entity_reference_f1=("reference_f1", "mean"),
        entity_source_support=("source_supported_precision", "mean"),
    )
    .reset_index()
)

percent_metrics = ["style_accuracy", "editorial_compliance", "entity_source_support"]
paper_table = grouped.copy()
paper_table[percent_metrics] = 100 * paper_table[percent_metrics]
paper_table.to_csv(ARTIFACTS_DIR / "table_01_metrics_by_platform.csv", index=False)
display(paper_table.round(3))

## 3. Bootstrap confidence intervals

In [ ]:
def bootstrap_mean_ci(values, seed, n_resamples=N_BOOTSTRAP):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = np.empty(n_resamples)
    batch = 200
    for start in range(0, n_resamples, batch):
        stop = min(start + batch, n_resamples)
        samples = rng.choice(values, size=(stop - start, len(values)), replace=True)
        means[start:stop] = samples.mean(axis=1)
    return float(values.mean()), *np.quantile(means, [0.025, 0.975]).tolist()

ci_rows = []
for group_number, ((platform, system), group) in enumerate(
    results.groupby(["social_network", "system"], observed=True), start=1
):
    for metric in ["style_target_correct", "bertscore_f1_source", "all_editorial_constraints_ok"]:
        estimate, low, high = bootstrap_mean_ci(group[metric], RANDOM_SEED + group_number)
        ci_rows.append({
            "social_network": platform,
            "system": system,
            "metric": metric,
            "estimate": estimate,
            "ci95_low": low,
            "ci95_high": high,
            "n": len(group),
        })

confidence_intervals = pd.DataFrame(ci_rows)
confidence_intervals.to_csv(ARTIFACTS_DIR / "table_02_bootstrap_confidence_intervals.csv", index=False)
display(confidence_intervals.round(4))

## 4. Rankings within each platform

In [ ]:
metric_directions = {
    "style_accuracy": "max",
    "editorial_compliance": "max",
    "bertscore_f1": "max",
    "sentence_bleu": "max",
    "perplexity_median": "min",
    "entity_reference_f1": "max",
    "entity_source_support": "max",
}

winner_rows = []
for platform, platform_data in grouped.groupby("social_network", observed=True):
    for metric, direction in metric_directions.items():
        idx = platform_data[metric].idxmax() if direction == "max" else platform_data[metric].idxmin()
        winner = platform_data.loc[idx]
        winner_rows.append({
            "social_network": platform,
            "metric": metric,
            "best_system": winner["system"],
            "value": winner[metric],
            "direction": direction,
        })

winners = pd.DataFrame(winner_rows)
winners.to_csv(ARTIFACTS_DIR / "table_03_platform_winners.csv", index=False)
display(winners.pivot(index="metric", columns="social_network", values="best_system"))

heatmap_metrics = ["style_accuracy", "editorial_compliance", "bertscore_f1", "entity_source_support"]
fig, axes = plt.subplots(1, len(heatmap_metrics), figsize=(15, 5), sharey=True)
for ax, metric in zip(axes, heatmap_metrics):
    matrix = grouped.pivot(index="system", columns="social_network", values=metric)
    sns.heatmap(matrix, annot=True, fmt=".3f", cmap="viridis", ax=ax, cbar=False)
    ax.set(title=metric.replace("_", " ").title(), xlabel="Platform", ylabel="")
axes[0].set_ylabel("System")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "figure_01_metric_heatmaps.png")
plt.show()
plt.close(fig)

## 5. Platform gaps and adjusted tests

In [ ]:
def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * len(p) / np.arange(1, len(p) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    output = np.empty_like(adjusted)
    output[order] = np.clip(adjusted, 0, 1)
    return output

platforms = sorted(results["social_network"].unique())
tests = []
if len(platforms) == 2:
    first, second = platforms
    for system, system_data in results.groupby("system", observed=True):
        a = system_data.loc[system_data["social_network"] == first]
        b = system_data.loc[system_data["social_network"] == second]

        a_success = int(a["style_target_correct"].sum())
        b_success = int(b["style_target_correct"].sum())
        odds_ratio, p_style = stats.fisher_exact([
            [a_success, len(a) - a_success],
            [b_success, len(b) - b_success],
        ])
        tests.append({
            "system": system,
            "metric": "style_accuracy",
            "contrast": f"{first} - {second}",
            "difference": a["style_target_correct"].mean() - b["style_target_correct"].mean(),
            "effect": odds_ratio,
            "effect_name": "odds_ratio",
            "p_value": p_style,
        })

        x = a["bertscore_f1_source"].dropna().to_numpy()
        y = b["bertscore_f1_source"].dropna().to_numpy()
        mw = stats.mannwhitneyu(x, y, alternative="two-sided")
        rank_biserial = 2 * mw.statistic / (len(x) * len(y)) - 1
        tests.append({
            "system": system,
            "metric": "bertscore_f1_source",
            "contrast": f"{first} - {second}",
            "difference": x.mean() - y.mean(),
            "effect": rank_biserial,
            "effect_name": "rank_biserial_r",
            "p_value": mw.pvalue,
        })

tests = pd.DataFrame(tests)
if not tests.empty:
    tests["p_adjusted_bh"] = benjamini_hochberg(tests["p_value"])
    tests["significant_at_0.05"] = tests["p_adjusted_bh"] < 0.05
tests.to_csv(ARTIFACTS_DIR / "table_04_platform_gap_tests.csv", index=False)
display(tests.round(5))

gap_metrics = ["style_accuracy", "editorial_compliance", "bertscore_f1", "sentence_bleu", "entity_source_support"]
if len(platforms) == 2:
    wide = grouped.pivot(index="system", columns="social_network", values=gap_metrics)
    gap_rows = []
    for metric in gap_metrics:
        difference = wide[(metric, platforms[0])] - wide[(metric, platforms[1])]
        for system, value in difference.items():
            gap_rows.append({"system": system, "metric": metric, "difference": value})
    gaps = pd.DataFrame(gap_rows)
    gaps.to_csv(ARTIFACTS_DIR / "table_05_platform_gaps.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=gaps, y="system", x="difference", hue="metric", ax=ax)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set(
        title=f"Platform gap: {platforms[0]} minus {platforms[1]}",
        xlabel="Difference (positive values favor the first platform)", ylabel="System",
    )
    ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(ARTIFACTS_DIR / "figure_02_platform_gaps.png")
    plt.show()
    plt.close(fig)

## 6. Data-driven subgroup conclusions

In [ ]:
lines = []
for platform in sorted(grouped["social_network"].unique()):
    p = grouped[grouped["social_network"] == platform]
    style_best = p.loc[p["style_accuracy"].idxmax()]
    semantic_best = p.loc[p["bertscore_f1"].idxmax()]
    support_best = p.loc[p["entity_source_support"].idxmax()]
    if LANGUAGE == "EN":
        lines.append(
            f"On **{platform}**, **{style_best['system']}** has the highest style accuracy "
            f"({style_best['style_accuracy']:.1%}); **{semantic_best['system']}** has the highest mean "
            f"source BERTScore F1 ({semantic_best['bertscore_f1']:.3f}); and "
            f"**{support_best['system']}** has the strongest mean entity support "
            f"({support_best['entity_source_support']:.1%})."
        )
    else:
        lines.append(
            f"On **{platform}**, **{style_best['system']}** has the highest style accuracy "
            f"({style_best['style_accuracy']:.1%}); **{semantic_best['system']}** has the highest mean source BERTScore F1 "
            f"({semantic_best['bertscore_f1']:.3f}); and "
            f"**{support_best['system']}** has the strongest mean entity support "
            f"({support_best['entity_source_support']:.1%})."
        )

if len(platforms) == 2:
    style_gaps = gaps[gaps["metric"] == "style_accuracy"].copy()
    largest = style_gaps.loc[style_gaps["difference"].abs().idxmax()]
    if LANGUAGE == "EN":
        lines.append(
            f"The largest absolute style-accuracy gap is observed for **{largest['system']}** "
            f"({largest['difference']:+.1%} for {platforms[0]} minus {platforms[1]})."
        )
    else:
        lines.append(
            f"The largest absolute style-accuracy gap is observed for **{largest['system']}** "
            f"({largest['difference']:+.1%} for {platforms[0]} minus {platforms[1]})."
        )

if LANGUAGE == "EN":
    lines.append(
        "These subgroup results are descriptive rather than causal. Platform sample sizes are unequal, "
        "and topic/source composition can explain part of the observed gaps; confidence intervals and "
        "BH-adjusted tests should therefore accompany point estimates."
    )
else:
    lines.append(
        "These subgroup results are descriptive rather than causal. Platform sample sizes are unequal, "
        "and topic/source composition can explain part of the observed gaps; confidence intervals and "
        "BH-adjusted tests should therefore accompany point estimates."
    )

display(Markdown("\n".join(f"- {line}" for line in lines)))

summary_payload = {
    "language": LANGUAGE,
    "systems": sorted(results["system"].unique().tolist()),
    "platform_sample_sizes": {str(k): int(v) for k, v in test_lookup["social_network"].value_counts().items()},
    "best_systems_by_platform_and_metric": winners.to_dict(orient="records"),
}
(ARTIFACTS_DIR / "subgroup_summary.json").write_text(
    json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Saved subgroup tables, figures, and summary to {ARTIFACTS_DIR}")